In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from model.config.settings import TICKERS, START_DATE, END_DATE

In [ ]:
from model.src.data_pipeline import fetch_data
data_dict = {}
for ticker in TICKERS:
    df = fetch_data(ticker, START_DATE, END_DATE)
    data_dict[ticker] = df
    print(f'\n{ticker} Head:')
    print(df.head())
    print(f'{ticker} Shape: {df.shape}')

### Plot raw closing prices for all tickers

In [ ]:
plt.figure(figsize=(14, 7))
for ticker, df in data_dict.items():
    plt.plot(df.index, df['Close'], label=ticker)
plt.title('Raw Closing Prices')
plt.legend()
plt.show()

### Compute and visualize technical indicators

In [ ]:
from model.src.data_pipeline import preprocess
processed_dict = {ticker: preprocess(df) for ticker, df in data_dict.items()}
df_rel = processed_dict['RELIANCE.NS']

plt.figure(figsize=(14, 7))
plt.plot(df_rel.index, df_rel['Close'], label='Close')
plt.plot(df_rel.index, df_rel['SMA_20'], label='SMA 20')
plt.plot(df_rel.index, df_rel['SMA_50'], label='SMA 50')
plt.title('RELIANCE.NS Technical Indicators')
plt.legend()
plt.show()

### Correlation heatmap of features

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(df_rel.corr(), annot=False, cmap='coolwarm')
plt.title('Feature Correlation Heatmap')
plt.show()

### Distribution plots of daily returns per ticker

In [ ]:
plt.figure(figsize=(10, 6))
for ticker, df in processed_dict.items():
    sns.kdeplot(df['Daily_Return'].dropna(), label=ticker)
plt.title('Distribution of Daily Returns')
plt.legend()
plt.show()

### Stationarity check (ADF test) on closing prices

In [ ]:
for ticker, df in data_dict.items():
    result = adfuller(df['Close'].dropna())
    print(f'{ticker} ADF Statistic: {result[0]}')
    print(f'{ticker} p-value: {result[1]}')